In [8]:
import pandas as pd
import numpy as np

# Load the master harmonized dataset created in the previous step
df = pd.read_csv('../Data/data_processed/BRFSS_cancer_master_harmonized.csv')

print(f"Dataset successfully loaded. Initial shape: {df.shape}")

Dataset successfully loaded. Initial shape: (117280, 32)


In [2]:
# STEP 2: INSPECT CNCRTYP (CANCER TYPE)

# Check value distribution for CNCRTYP overall
print("\n=== Unique values of CNCRTYP overall ===")
print(df['CNCRTYP'].value_counts(dropna=False))

# Check CNCRTYP distribution across years to verify structural consistency
print("\n=== CNCRTYP distribution by Year ===")
print(pd.crosstab(df['YEAR'], df['CNCRTYP'], dropna=False))


=== Unique values of CNCRTYP overall ===
CNCRTYP
22.0    33248
1.0     10017
21.0     8874
5.0      7082
16.0     6614
30.0     6612
19.0     5640
20.0     4670
23.0     4348
77.0     4312
10.0     3094
2.0      2475
7.0      1825
3.0      1487
14.0     1423
8.0      1422
25.0     1399
24.0     1318
6.0      1301
18.0     1246
15.0     1225
28.0     1141
26.0     1015
4.0      1009
29.0      980
17.0      920
12.0      817
99.0      572
27.0      468
13.0      345
11.0      184
NaN       120
9.0        77
Name: count, dtype: int64

=== CNCRTYP distribution by Year ===
CNCRTYP  1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0  ...  \
YEAR                                                                 ...   
2014     1190   347   184   106    20    22    46   133     5   343  ...   
2016     1251   342   188   102    20    24    36   132     9   315  ...   
2017     1204   253   158   121    26    30    35   152     5   307  ...   
2018      911   227   132    75    16    21 

In [3]:
# STEP 3: SYSTEMATIC VALUE CHECK FOR ALL COLUMNS

# Automatically scan all columns to detect categorical distribution or numerical ranges (min/max),
# helping identify potential special CDC codes (e.g., 'Don't know', 'Refused').

print("\n=== Systematic Value Check across all columns ===")
for col in df.columns:
    unique_vals = df[col].dropna().unique()
    
    # If the column has a small number of unique values, print them sorted
    if len(unique_vals) <= 15:
        sorted_vals = sorted([str(x) for x in unique_vals])
        print(f"Column: {col} -> Categorical/Small unique values ({len(unique_vals)}): {sorted_vals}")
    else:
        # For continuous or high-cardinality columns, print min and max bounds
        print(f"Column: {col} -> Continuous/High cardinality ({len(unique_vals)} values) | Min: {df[col].min()} | Max: {df[col].max()}")


=== Systematic Value Check across all columns ===
Column: GENHLTH -> Categorical/Small unique values (7): ['1.0', '2.0', '3.0', '4.0', '5.0', '7.0', '9.0']
Column: PHYSHLTH -> Continuous/High cardinality (33 values) | Min: 1.0 | Max: 99.0
Column: MENTHLTH -> Continuous/High cardinality (33 values) | Min: 1.0 | Max: 99.0
Column: POORHLTH -> Continuous/High cardinality (33 values) | Min: 1.0 | Max: 99.0
Column: CNCRAGE -> Continuous/High cardinality (99 values) | Min: 1.0 | Max: 99.0
Column: CNCRTYP -> Continuous/High cardinality (32 values) | Min: 1.0 | Max: 99.0
Column: CSRVTRT -> Categorical/Small unique values (7): ['1.0', '2.0', '3.0', '4.0', '5.0', '7.0', '9.0']
Column: CSRVDOC1 -> Categorical/Small unique values (12): ['1.0', '10.0', '2.0', '3.0', '4.0', '5.0', '6.0', '7.0', '77.0', '8.0', '9.0', '99.0']
Column: CSRVPAIN -> Categorical/Small unique values (4): ['1.0', '2.0', '7.0', '9.0']
Column: _AGE80 -> Continuous/High cardinality (63 values) | Min: 18.0 | Max: 80.0
Column: SE

In [14]:
# STEP 4: MISSING VALUES & SPECIAL CODES STRATEGY (Based on Codebook Rules)

#df = pd.read_csv('../Data/data_processed/BRFSS_cancer_master_harmonized.csv')

# 2. Recode CDC special codes (7, 77, 777, 9, 99, 999) to NaN
special_missing_codes = [7.0, 77.0, 777.0, 9.0, 99.0, 999.0]
cols_to_clean = [
    'GENHLTH', 'CNCRTYP', 'CSRVTRT', 'CSRVDOC1', 'CSRVPAIN', 
    'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ADDEPEV', 'DIABETE', 
    'HLTHPLN', 'PERSDOC', 'MEDCOST_U', 'INCOME', 'EDUCA', 
    '_RACE', 'MARITAL', 'CHECKUP1', 'SMOKE100', 'EXERANY2', 
    'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'SLEPTIM1', 'DRVISITS', 'SEX'
]

for col in cols_to_clean:
    if col in df.columns:
        df[col] = df[col].replace(special_missing_codes, np.nan)

# 3. Handle PHYSHLTH and MENTHLTH: Recode 88 -> 0 (None / 0 days of bad health)
for col in ['PHYSHLTH', 'MENTHLTH']:
    if col in df.columns:
        df[col] = df[col].replace(88.0, 0.0)

# 4. Handle POORHLTH: Recode 88 -> 0 AND apply skip logic (if physical/mental health are 0, missing poor health = 0)
if 'POORHLTH' in df.columns:
    df['POORHLTH'] = df['POORHLTH'].replace(88.0, 0.0)
    skip_condition = (df['PHYSHLTH'] == 0.0) & (df['MENTHLTH'] == 0.0) & (df['POORHLTH'].isna())
    df.loc[skip_condition, 'POORHLTH'] = 0.0

# 5. Handle Cancer Treatment Skip Logics (CSRVDOC1, CSRVPAIN, CSRVTRT) -> Recode BLANK to -1 (Not applicable)
treatment_skip_cols = ['CSRVDOC1', 'CSRVPAIN', 'CSRVTRT']
for col in treatment_skip_cols:
    if col in df.columns:
        df[col] = df[col].fillna(-1.0)

# 6. Drop rows with negligible missing values in key clinical variables
key_cols_to_drop_na = ['CNCRTYP', 'INCOME', '_RACE', 'PHYSHLTH', 'CVDINFR4']
df = df.dropna(subset=[col for col in key_cols_to_drop_na if col in df.columns])

print(f"Dataset shape after Missing Values Strategy: {df.shape}")

# STEP 5: ADVANCED COVID PHASE FEATURE ENGINEERING

# Define the exact periods based on our refined methodology
pre = [2014, 2016, 2017, 2018, 2019]
peak = [2020, 2021]
post = [2022, 2023, 2024]

# Function to map each year to its corresponding pandemic phase
def assign_covid_phase(year):
    # Ensure year is treated as integer
    year_int = int(year)
    if year_int in pre:
        return 'Pre-Pandemic'
    elif year_int in peak:
        return 'Pandemic Peak'
    elif year_int in post:
        return 'Pandemic Recovery'
    else:
        return 'Unknown'

# Apply the mapping to create the new feature
if 'YEAR' in df.columns:
    df['COVID_PHASE'] = df['YEAR'].apply(assign_covid_phase)

# Verify the distribution of cases across the refined phases
print("\n=== Cancer Cases Distribution Across Refined COVID Phases ===")
phase_counts = df['COVID_PHASE'].value_counts()
print(phase_counts)

# Also keep a binary or numeric mapping if needed for machine learning models later
# Pre-Pandemic = 0, Pandemic Peak = 1, Pandemic Recovery = 2 (Ordinal mapping)
phase_mapping = {'Pre-Pandemic': 0, 'Pandemic Peak': 1, 'Pandemic Recovery': 2}
df['COVID_PHASE_CODE'] = df['COVID_PHASE'].map(phase_mapping)

print("\nRefined COVID phase features successfully engineered!")


# STEP 6: SAVE THE FINAL ANALYSIS-READY DATASET
import os

# Define output directory and file path
output_dir = '../Data/data_processed'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'BRFSS_cancer_analysis_ready.csv')

# Export dataframe to CSV (without index)
df.to_csv(output_path, index=False)

print(f"Final analysis-ready dataset successfully saved to: {output_path}")
print(f"Final dataset shape: {df.shape}")

Dataset shape after Missing Values Strategy: (62314, 34)

=== Cancer Cases Distribution Across Refined COVID Phases ===
COVID_PHASE
Pandemic Recovery    24956
Pre-Pandemic         19279
Pandemic Peak        18079
Name: count, dtype: int64

Refined COVID phase features successfully engineered!
Final analysis-ready dataset successfully saved to: ../Data/data_processed\BRFSS_cancer_analysis_ready.csv
Final dataset shape: (62314, 34)
